# PCA 主成分分析通用模板 · Python 3.11 · 美赛直接复用

**适用**：多指标降维、消除共线、构造综合评价指标、样本排序与可视化。

## 模型与论文可引用的公式

1. **标准化**：$z_{ij}=\dfrac{x_{ij}-\bar x_j}{s_j}$，消除量纲；
   PCA 通常基于**相关矩阵** $R=\dfrac{1}{n-1}Z^\top Z$ 而非协方差矩阵。
2. **适用性检验**：KMO 越接近 1 越适合降维（>0.7 较好）；
   Bartlett 球形检验 $\chi^2=-\left[n-1-\frac{2m+5}{6}\right]\ln|R|$，
   $df=\frac{m(m-1)}{2}$，$p<0.05$ 说明变量间存在相关，适合做 PCA。
3. **特征值分解**：$R u_j=\lambda_j u_j$，按 $\lambda_j$ 降序排列；
   **贡献率** $=\lambda_j/\sum_k\lambda_k$，**累计贡献率** $=\sum_{k\le j}\lambda_k/\sum_k\lambda_k$。
4. **载荷** $a_{ij}=u_{ij}\sqrt{\lambda_j}$（变量与主成分的相关系数）；
   **主成分得分** $F_j=Z u_j$；**变量共同度** $h_i^2=\sum_{j=1}^{m}a_{ij}^2$。
5. **选主成分个数**：Kaiser 准则（$\lambda>1$）+ 累计贡献率 ≥ 80%（可配）。
6. **综合得分**（美赛常用）：$F=\sum_{j=1}^{m}w_jF_j$，$w_j=\lambda_j/\sum_{k\le m}\lambda_k$，
   据此给样本排名。

> **注意**：主成分的正负方向不唯一（特征向量可整体取反），本模板固定「绝对值最大的载荷为正」
> 保证可复现；综合得分「越大越好」默认按第一主成分载荷和为正来定向，
> 若题目里存在极小型指标，应先用评价类模型做正向化再跑 PCA。
> 独热编码产生的二值列会让 KMO/Bartlett 结果仅供参考。

In [2]:
# ============================================================

# 0. 环境准备（Python 3.11）

# 本 Notebook 自包含：不 import 其他 Notebook，换题目直接整份复制走

# ============================================================

import sys

import warnings

import platform

from pathlib import Path

from dataclasses import dataclass, field, fields, replace



import numpy as np

import pandas as pd

import matplotlib

import matplotlib.pyplot as plt

from scipy import stats

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster



from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.decomposition import PCA as SkPCA

from sklearn.cluster import KMeans

from sklearn.linear_model import LogisticRegression, LinearRegression

from sklearn.metrics import (

    silhouette_score, silhouette_samples,

    calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score,

    accuracy_score, precision_score, recall_score, f1_score,

    roc_auc_score, roc_curve, precision_recall_curve,

    confusion_matrix, classification_report, ConfusionMatrixDisplay,

)

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

from sklearn.base import clone

from pandas.api.types import is_numeric_dtype, is_datetime64_any_dtype



if sys.version_info[:2] < (3, 11):

    print(f"[警告] 当前 Python {platform.python_version()}，本模板按 3.11 编写。")

warnings.filterwarnings("ignore", category=FutureWarning)

warnings.filterwarnings("ignore", category=UserWarning)



plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "SimSun", "Arial Unicode MS"]

plt.rcParams["axes.unicode_minus"] = False

plt.rcParams["figure.dpi"] = 110

plt.rcParams["axes.grid"] = True

plt.rcParams["grid.alpha"] = 0.3

pd.set_option("display.unicode.east_asian_width", True)

pd.set_option("display.max_columns", 40)

pd.set_option("display.width", 160)

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")



import sklearn

print("Python", platform.python_version(), "| numpy", np.__version__,

      "| pandas", pd.__version__, "| sklearn", sklearn.__version__)

Python 3.11.9 | numpy 2.4.6 | pandas 3.1.0 | sklearn 1.9.1


### 换一份数据，怎么复用？（照抄即可）



1. 把新文件放进 `导入数据/`（`.csv` / `.xlsx` / `.xls` 都支持）；

2. 在 Notebook 末尾新建一格，只写一行：



```python

bundle = run_on("你的文件名.csv", target=None, tag="本次名字")

```



3. `Run All`。结果自动写进 `输出数据/`，文件名形如

`PCA_示例_结果汇总.xlsx`（前缀固定，中间是 tag）：CSV 明细 + 多 Sheet Excel + 图片。



**数据格式**：一行一个样本、一列一个指标，中文列名没问题。BOM、前后空格、

文本数字、`999` 哨兵、重复行、缺失值、标准化、文字类别转独热编码全部自动处理。



**只有 3 个值要填**：文件名、`target`（`填 None（PCA 是无监督方法，不需要标签）`）、`tag`（换数据就改名，否则覆盖上一份）。

其余参数（`id_cols` / `features` / `na_markers` / `scale` / `k_max` 等）只在需要时改。

`run_on` 也接受任意 Config 字段作关键字参数，例如 `make_plots=False, k_max=10`。



> 输出文件名固定、同名覆盖：再跑 10 次也不会多出文件；换了 `tag` 才会新增一组。

> 想跳过末尾示例就不动 `RUN_EXAMPLES=False`；想一张图都不生成用 `make_plots=False`；

> 清旧结果用 `clean_outputs("旧tag")`。**重新打开 Notebook 才会读到磁盘上的最新版。**

In [4]:
# ============================================================

# 1. 路径 + 配置区（换数据只改这里，或直接用后面的 run_on 一行搞定）

# ============================================================

PREFIX = "PCA"          # 本 Notebook 所有输出文件的固定前缀



def find_project_root(marker="导入数据"):

    """从当前目录向上找含 导入数据/ 的项目根，兼容从子目录或根目录启动内核。"""

    for base in [Path.cwd(), *Path.cwd().parents]:

        if (base / marker).is_dir():

            return base

    return Path.cwd()



ROOT = find_project_root()

DATA_DIR = ROOT / "导入数据" if (ROOT / "导入数据").is_dir() else ROOT

OUT_DIR = ROOT / "输出数据" if (ROOT / "输出数据").is_dir() else ROOT

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("项目根目录 :", ROOT)

print("输出目录   :", OUT_DIR)





@dataclass

class Config:

    """所有建模开关集中在这里；标 ★ 的是换数据时通常要改的。"""

    data_path: object = None        # ★ 数据文件；None = 自动找 导入数据/ 里第一个表格

    sheet_name: object = 0          # Excel 工作表（名字或序号）

    id_cols: tuple = ("date",)      # ★ 编号/日期等不进模型的列

    target: object = None           # ★ 有监督标签列；None = 无监督（KMeans/PCA 不填）

    features: object = None         # 参与建模的列；None = 自动推断

    tag: str = "demo"               # ★ 输出文件名前缀，换数据记得改名

    make_plots: bool = True         # False = 一张图都不生成（连画布都不创建）



    # ---- 读入与清洗 ----

    na_markers: tuple = (999, 999.0, -999, "未填", "NA", "N/A", "null", "未知")

    drop_duplicates: bool = True

    drop_high_missing: bool = True

    missing_threshold: float = 0.50  # 缺失率超过它就不进模型



    # ---- 预处理 ----

    impute_num: str = "median"      # median / mean / zero

    impute_cat: str = "mode"        # mode / unknown

    add_missing_indicator: bool = True

    outlier: str = "winsor_iqr"     # winsor_iqr / winsor_3sigma / none

    scale: str = "standard"         # standard / minmax / robust / none

    max_onehot_cardinality: int = 30

    onehot_drop_first: bool = False

    random_state: int = 42          # 全局随机种子，保证结果可复现


    # ---- PCA 专有 ----
    pca_var_threshold: float = 0.80  # 累计贡献率阈值
    pca_max_components: int = 10     # 最多保留的主成分数
    pca_flip_auto: bool = True       # 自动调整第一主成分方向，使综合得分「越大越好」




CONFIG = Config(data_path=DATA_DIR / "mcm_final_raw.csv", target=None, tag="示例")

RUN_EXAMPLES = False                # True 才会执行末尾的可选示例（多跑一组数据、多产一组文件）





def save_fig(fig, name, cfg):

    """统一保存图片：文件名 = {PREFIX}_{tag}_{图名}.png，同名覆盖；用完关闭画布。"""

    if not getattr(cfg, "make_plots", True):

        plt.close(fig)

        return None

    path = OUT_DIR / f"{PREFIX}_{cfg.tag}_{name}.png"

    fig.savefig(path, dpi=150, bbox_inches="tight")

    plt.show()

    plt.close(fig)                  # 释放 figure，反复 Run All 不会堆积内存

    print(f"[图片] {path.name}（同名覆盖）")

    return path





def clean_outputs(tag=None):

    """清理某个 tag 遗留的产物（只匹配本模板前缀，不碰其他文件）。"""

    tag = CONFIG.tag if tag is None else tag

    files = sorted(OUT_DIR.glob(f"{PREFIX}_{tag}_*"))

    for f in files:

        f.unlink()

    print(f"已清理 tag='{tag}' 的 {len(files)} 个文件")

    return files





print("配置：", CONFIG)

print("运行末尾示例：", RUN_EXAMPLES)

项目根目录 : D:\git项目
输出目录   : D:\git项目\输出数据
配置： Config(data_path=WindowsPath('D:/git项目/导入数据/mcm_final_raw.csv'), sheet_name=0, id_cols=('date',), target=None, features=None, tag='示例', make_plots=True, na_markers=(999, 999.0, -999, '未填', 'NA', 'N/A', 'null', '未知'), drop_duplicates=True, drop_high_missing=True, missing_threshold=0.5, impute_num='median', impute_cat='mode', add_missing_indicator=True, outlier='winsor_iqr', scale='standard', max_onehot_cardinality=30, onehot_drop_first=False, random_state=42, pca_var_threshold=0.8, pca_max_components=10, pca_flip_auto=True)
运行末尾示例： False


### 2. 通用数据读取 `load_table`



自动处理美赛附件的常见坑：CSV 编码/BOM、列名与字符串首尾空格、数字被存成文本、

`999` 等哨兵值、日期列、全空行列、重复行，并打印每一步改了什么（论文预处理段落可直接引用）。

In [6]:
TEXT_NA = ["", " ", "  ", "-", "?", "NA", "N/A", "na", "n/a", "null", "NULL",

           "None", "nan", "NaN", "未填", "未知", "缺失", "无"]





def resolve_data_path(cfg):

    """确定数据文件；相对路径按 导入数据/ 解析，没指定就自动挑一个表格。"""

    if cfg.data_path is not None:

        p = Path(cfg.data_path)

        if not p.is_absolute():

            p = (ROOT / p) if (p.parts and p.parts[0] == DATA_DIR.name) else (DATA_DIR / p)

        if not p.exists():

            raise FileNotFoundError(f"找不到数据文件：{p}\n请检查文件名或传入绝对路径。")

        return p

    cand = sorted([p for p in DATA_DIR.iterdir()

                   if p.suffix.lower() in (".csv", ".xlsx", ".xls")])

    if not cand:

        raise FileNotFoundError(f"{DATA_DIR} 里没有 csv/xlsx 文件。")

    print(f"[提示] 自动选用：{cand[0].name}")

    return cand[0]





def load_table(cfg):

    """读入 → 初清洗 → (DataFrame, 元信息)。数值哨兵留到 3.5 步替换以便统计个数。"""

    path = resolve_data_path(cfg)

    na_values = list(dict.fromkeys(TEXT_NA))

    if path.suffix.lower() in (".xlsx", ".xls"):

        df = pd.read_excel(path, sheet_name=cfg.sheet_name, na_values=na_values)

        enc = "Excel(openpyxl)"

    else:

        df, enc = None, None

        for e in ("utf-8-sig", "utf-8", "gbk", "gb18030"):

            try:

                df = pd.read_csv(path, encoding=e, na_values=na_values,

                                 skipinitialspace=True)

                enc = e

                break

            except UnicodeDecodeError:

                continue

        if df is None:

            raise UnicodeDecodeError("all", b"", 0, 1, "常见编码都无法解码该 CSV。")



    n_raw = len(df)

    # 列名：去 BOM / 去空格 / 重名加后缀

    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]

    seen, cols = {}, []

    for c in df.columns:

        seen[c] = seen.get(c, 0) + 1

        cols.append(c if seen[c] == 1 else f"{c}_{seen[c]}")

    df.columns = cols

    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all").copy()



    # 字符串列：strip 后把文字型缺失标记置 NaN

    for c in df.columns:

        if not is_numeric_dtype(df[c]) and not is_datetime64_any_dtype(df[c]):

            df[c] = df[c].map(lambda v: v.strip() if isinstance(v, str) else v)

            df[c] = df[c].replace({m: np.nan for m in TEXT_NA})



    # 文本数字列转数值；名字带 date/time 且能解析的列转 datetime

    turned, date_cols = [], []

    for c in df.columns:

        if is_numeric_dtype(df[c]) or is_datetime64_any_dtype(df[c]):

            continue

        s = df[c]

        valid = s.notna().sum()

        if valid == 0:

            continue

        conv = pd.to_numeric(s, errors="coerce")

        if conv.notna().sum() >= 0.90 * valid:

            df[c] = conv.astype(float)

            turned.append(c)

        elif any(k in c.lower() for k in ("date", "time", "日期", "时间")):

            dt = pd.to_datetime(s, errors="coerce")

            if dt.notna().sum() >= 0.90 * valid:

                df[c] = dt

                date_cols.append(c)



    # 数值哨兵（999 等）转 NaN，逐列统计，便于论文说明处理依据

    hits = {}

    num_sentinels = [v for v in cfg.na_markers if isinstance(v, (int, float))]

    for c in df.select_dtypes(include="number").columns:

        hit = int(df[c].isin(num_sentinels).sum())

        if hit:

            hits[c] = hit

            df[c] = df[c].replace({v: np.nan for v in num_sentinels})



    n_dup = int(df.duplicated().sum())

    if cfg.drop_duplicates and n_dup:

        df = df.drop_duplicates().reset_index(drop=True)



    meta = dict(path=path, encoding=enc, n_raw=n_raw, n_final=len(df),

                n_duplicates=n_dup, numeric_turned=turned,

                date_cols=date_cols, sentinel_hits=hits)

    print(f"读取成功 : {path.name}（{enc}）")

    print(f"数据规模 : {n_raw} 行 → {len(df)} 行，{df.shape[1]} 列；删重复 {n_dup} 行")

    print(f"文本转数值 : {turned or '无'}；日期列：{date_cols or '无'}；哨兵转缺失：{hits or '无'}")

    return df, meta





df, META = load_table(CONFIG)

print("\n字段类型：")

print(df.dtypes.to_string())

读取成功 : mcm_final_raw.csv（utf-8-sig）
数据规模 : 381 行 → 366 行，8 列；删重复 15 行
文本转数值 : ['pm25']；日期列：['date']；哨兵转缺失：{'traffic_index': 6}

字段类型：
date                datetime64[us]
region                         str
pm25                       float64
temperature                float64
traffic_index              float64
fuel_type                      str
survey_score               float64
household_income           float64


### 3. 数据体检



建模前先看每列的缺失率、唯一值、极值与 IQR 异常点；缺失率过高的列会在预处理阶段自动剔除。

一张综合图（缺失率 + 标准化箱线图）便于写进论文的数据说明部分。

In [8]:
def data_audit(df, cfg, plot=True):

    """逐列体检表；plot=True 且 make_plots=True 时画一张综合图。"""

    rows = []

    for c in df.columns:

        s = df[c]

        miss = int(s.isna().sum())

        rec = dict(列名=c, 类型=str(s.dtype), 缺失数=miss,

                   缺失率=round(miss / len(df), 4), 唯一值=int(s.nunique(dropna=True)))

        if is_numeric_dtype(s):

            q1, q3 = s.quantile(0.25), s.quantile(0.75)

            iqr = q3 - q1

            rec.update(最小值=round(float(s.min()), 3) if s.notna().any() else np.nan,

                       中位数=round(float(s.median()), 3) if s.notna().any() else np.nan,

                       最大值=round(float(s.max()), 3) if s.notna().any() else np.nan,

                       IQR异常点=int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()),

                       建议=("剔除(缺测过多)" if (cfg.drop_high_missing and

                                             miss / len(df) > cfg.missing_threshold)

                             else ("常数列,剔除" if s.nunique(dropna=True) <= 1 else "保留")))

        else:

            rec.update(最小值="", 中位数="", 最大值="", IQR异常点=0,

                       建议=("日期/标识列,不建模" if is_datetime64_any_dtype(s)

                             else "独热编码" if s.nunique(dropna=True) <= cfg.max_onehot_cardinality

                             else "高基数,剔除"))

        rows.append(rec)

    audit = pd.DataFrame(rows)

    print("===== 数据体检表 =====")

    print(audit.to_string(index=False))



    if plot and cfg.make_plots:

        num = [c for c in df.columns if is_numeric_dtype(df[c])]

        miss_tbl = audit[audit["缺失率"] > 0]

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

        if len(miss_tbl):

            axes[0].barh(miss_tbl["列名"], miss_tbl["缺失率"] * 100, color="#d95f5f")

            axes[0].axvline(cfg.missing_threshold * 100, ls="--", color="#444")

        axes[0].set_title("各列缺失率 (%)"); axes[0].set_xlabel("%")

        if num:

            z = df[num].apply(lambda s: (s - s.mean()) / (s.std() or 1))

            axes[1].boxplot([z[c].dropna() for c in num])

            axes[1].set_xticks(range(1, len(num) + 1))

            axes[1].set_xticklabels(num, rotation=30, ha="right")

            axes[1].set_title("数值列分布（标准化后）")

        fig.tight_layout()

        save_fig(fig, "数据体检", cfg)

    return audit





audit = data_audit(df, CONFIG)

===== 数据体检表 =====
            列名           类型  缺失数  缺失率  唯一值     最小值     中位数     最大值  IQR异常点               建议
            date datetime64[us]       0  0.0000      90                                           0 日期/标识列,不建模
          region            str       0  0.0000       4                                           0           独热编码
            pm25        float64      18  0.0492     344    18.6900    50.4260    93.2070          1               保留
     temperature        float64       5  0.0137     356    11.8600    23.2560    30.6550          0               保留
   traffic_index        float64      15  0.0410     351    43.7890    98.9720   150.0030          4               保留
       fuel_type            str       8  0.0219       4                                           0           独热编码
    survey_score        float64      18  0.0492     342    44.0990    69.7360    99.6440          3               保留
household_income        float64     318  0.8689      42 3,115.6080 5,240.4430 9,9

### 4. 可复用预处理器 `TabularPreprocessor`



`fit` 只在训练数据上统计中位数/众数、缩尾边界、标准化参数、类别清单，

`transform` 对新数据用**同一套参数**，避免数据泄漏。自动剔除标签列、日期列、

高缺失列、常数列、高基数文本列；数值列填补 → 缩尾 → 标准化，文字列众数填补 → 独热编码。

In [10]:
class TabularPreprocessor:

    """任意表格 → 数值特征矩阵 X（保留可读列名），支持 fit/transform 复用。"""



    def __init__(self, cfg):

        self.cfg = cfg

        self.numeric_cols_, self.categorical_cols_ = [], []

        self.indicator_cols_, self.dropped_cols_ = [], {}

        self.fill_num_, self.fill_cat_, self.bounds_ = {}, {}, {}

        self.center_, self.scale_, self.categories_ = {}, {}, {}

        self.feature_names_ = []



    def _pick_columns(self, df):

        excluded = set(self.cfg.id_cols or ())

        if self.cfg.target is not None:

            excluded.add(self.cfg.target)

        for c in df.columns:

            if is_datetime64_any_dtype(df[c]) or any(

                    k in str(c).lower() for k in ("date", "time", "日期", "时间")):

                excluded.add(c)

        cols = [c for c in df.columns if c not in excluded]

        if self.cfg.features is not None:

            cols = [c for c in self.cfg.features if c in df.columns]

        for c in cols:

            s = df[c]

            r = s.isna().mean()

            if r == 1:

                self.dropped_cols_[c] = "全空列"; continue

            if self.cfg.drop_high_missing and r > self.cfg.missing_threshold:

                self.dropped_cols_[c] = f"缺失率 {r:.1%}"; continue

            if s.nunique(dropna=True) <= 1:

                self.dropped_cols_[c] = "常数列"; continue

            if is_numeric_dtype(s):

                self.numeric_cols_.append(c)

            elif s.nunique(dropna=True) > self.cfg.max_onehot_cardinality:

                self.dropped_cols_[c] = "高基数文本列"

            else:

                self.categorical_cols_.append(c)

        if not self.numeric_cols_ and not self.categorical_cols_:

            raise ValueError("没有可用特征，请检查 features / missing_threshold。")



    def fit(self, df):

        self._pick_columns(df)

        cfg = self.cfg

        for c in self.numeric_cols_:

            s = pd.to_numeric(df[c], errors="coerce").astype(float)

            fill = (float(s.mean()) if cfg.impute_num == "mean" else

                    0.0 if cfg.impute_num == "zero" else float(s.median()))

            self.fill_num_[c] = fill if np.isfinite(fill) else 0.0

            if s.isna().any() and cfg.add_missing_indicator and f"{c}_missing" not in df.columns:

                self.indicator_cols_.append(c)

            f = s.fillna(self.fill_num_[c])

            if cfg.outlier == "winsor_iqr":

                q1, q3 = f.quantile(0.25), f.quantile(0.75)

                i = q3 - q1

                lo, hi = float(q1 - 1.5 * i), float(q3 + 1.5 * i)

            elif cfg.outlier == "winsor_3sigma":

                mu, sd = float(f.mean()), float(f.std())

                lo, hi = mu - 3 * sd, mu + 3 * sd

            else:

                lo, hi = -np.inf, np.inf

            self.bounds_[c] = (lo, hi)

            z = f.clip(lo, hi)

            if cfg.scale == "minmax":

                self.center_[c], self.scale_[c] = float(z.min()), float(z.max() - z.min())

            elif cfg.scale == "robust":

                self.center_[c] = float(z.median())

                self.scale_[c] = float(z.quantile(0.75) - z.quantile(0.25))

            elif cfg.scale == "standard":

                self.center_[c], self.scale_[c] = float(z.mean()), float(z.std())

            else:

                self.center_[c], self.scale_[c] = 0.0, 1.0

            if not np.isfinite(self.scale_[c]) or self.scale_[c] == 0:

                self.scale_[c] = 1.0

        for c in self.categorical_cols_:

            s = df[c]

            if cfg.impute_cat == "unknown":

                self.fill_cat_[c] = "未知"

            else:

                m = s.mode(dropna=True)

                self.fill_cat_[c] = str(m.iat[0]) if len(m) else "未知"

            cats = sorted({str(v).strip() for v in s.dropna().unique()})

            if cfg.onehot_drop_first and len(cats) > 1:

                cats = cats[1:]

            self.categories_[c] = cats

        names = list(self.numeric_cols_) + [f"{c}_missing" for c in self.indicator_cols_]

        for c in self.categorical_cols_:

            names += [f"{c}={v}" for v in self.categories_[c]]

        seen, out = {}, []

        for n in names:

            seen[n] = seen.get(n, 0) + 1

            out.append(n if seen[n] == 1 else f"{n}_{seen[n]}")

        self.feature_names_ = out

        return self



    def transform(self, df):

        miss = [c for c in self.numeric_cols_ + self.categorical_cols_ if c not in df.columns]

        if miss:

            raise KeyError(f"新数据缺少建模所需列：{miss}")

        blocks = []

        for c in self.numeric_cols_:

            s = pd.to_numeric(df[c], errors="coerce").astype(float).fillna(self.fill_num_[c])

            lo, hi = self.bounds_[c]

            blocks.append(((s.clip(lo, hi) - self.center_[c]) / self.scale_[c]).to_numpy(float))

        for c in self.indicator_cols_:

            blocks.append(df[c].isna().to_numpy(float))

        for c in self.categorical_cols_:

            s = df[c].map(lambda v: str(v).strip() if pd.notna(v) else self.fill_cat_[c])

            for v in self.categories_[c]:

                blocks.append((s == v).to_numpy(float))

        return pd.DataFrame(np.column_stack(blocks), columns=self.feature_names_, index=df.index)



    def fit_transform(self, df):

        return self.fit(df).transform(df)





pre = TabularPreprocessor(CONFIG).fit(df)

X = pre.transform(df)

print("数值列 :", pre.numeric_cols_)

print("类别列 :", pre.categorical_cols_)

print("剔除列 :", pre.dropped_cols_ or "无")

print("特征矩阵 X :", X.shape)

数值列 : ['pm25', 'temperature', 'traffic_index', 'survey_score']
类别列 : ['region', 'fuel_type']
剔除列 : {'household_income': '缺失率 86.9%'}
特征矩阵 X : (366, 16)


### 5. 适用性检验、特征值分解与综合得分

KMO / Bartlett 判断是否适合降维；相关矩阵特征分解得贡献率、载荷与主成分得分，
再按贡献率加权得到综合得分与排名。

In [12]:
# ============================================================
# 5. 适用性检验 + 特征值分解 + 载荷 / 得分 / 综合评分
# ============================================================
def kmo_bartlett(X):
    """KMO 测度与 Bartlett 球形检验（基于相关矩阵与偏相关矩阵）。"""
    R = np.nan_to_num(np.corrcoef(X.to_numpy(float), rowvar=False))
    m = R.shape[0]
    inv = np.linalg.pinv(R)
    d = np.sqrt(np.clip(np.diag(inv), 1e-12, None))
    P = -inv / np.outer(d, d)
    np.fill_diagonal(P, 1.0)
    r, p = R.copy(), P.copy()
    np.fill_diagonal(r, 0.0); np.fill_diagonal(p, 0.0)
    kmo = float((r ** 2).sum() / ((r ** 2).sum() + (p ** 2).sum() + 1e-12))
    eig = np.linalg.eigvalsh(R)
    pos = eig[eig > 1e-8]
    r_eff = len(pos)                            # 有效秩：独热列完全共线时会 < m
    singular = r_eff < m
    ln_det = float(np.log(pos).sum()) if r_eff else -np.inf
    n = len(X)
    # 相关矩阵奇异时 ln|R| 无定义：改用有效秩计算卡方与自由度，并明确标注「仅供参考」
    chi2 = -(n - 1 - (2 * r_eff + 5) / 6) * ln_det
    dfree = r_eff * (r_eff - 1) / 2
    pv = float(stats.chi2.sf(chi2, dfree)) if np.isfinite(chi2) else 0.0
    res = dict(KMO=round(kmo, 4), Bartlett卡方=round(chi2, 2),
               自由度=int(dfree), p值=round(pv, 6),
               KMO判断=("适合" if kmo > 0.7 else "尚可" if kmo > 0.6 else "不太适合"),
               Bartlett判断=("拒绝变量独立，适合 PCA" if pv < 0.05 else "变量相关性不足"),
               有效秩=r_eff,
               说明=("相关矩阵奇异（存在完全共线的独热列），检验值仅供参考"
                     if singular else "相关矩阵满秩"))
    print("===== PCA 适用性检验 =====")
    print(pd.DataFrame([res]).to_string(index=False))
    if kmo < 0.6 or pv >= 0.05:
        print("[提示] 适用性检验未通过：论文里应如实说明，并考虑改用熵权法 / TOPSIS，"
              "或先看相关矩阵挑出真正相关的指标组再做 PCA。")
    return res


def fit_pca(X, cfg):
    """自实现 PCA：标准化 → 相关矩阵 → 特征值分解 → 载荷 / 得分，并与 sklearn 对照。"""
    V = X.to_numpy(float)
    mu = V.mean(axis=0)
    sd = V.std(axis=0, ddof=1)
    sd = np.where(sd < 1e-12, 1.0, sd)          # 常数列除零保护
    Z = (V - mu) / sd
    R = np.nan_to_num(np.corrcoef(Z, rowvar=False))
    m = R.shape[0]
    eigval, eigvec = np.linalg.eigh(R)
    idx = np.argsort(eigval)[::-1]
    eigval = np.clip(eigval[idx], 0.0, None)
    eigvec = eigvec[:, idx]
    # 符号约定：每个特征向量绝对值最大的分量取正，保证每次运行结果一致
    for j in range(m):
        if eigvec[np.argmax(np.abs(eigvec[:, j])), j] < 0:
            eigvec[:, j] *= -1
    contrib = eigval / eigval.sum()
    cum = np.cumsum(contrib)
    loading = eigvec * np.sqrt(eigval)
    score = Z @ eigvec
    # 综合得分方向：第一主成分载荷和为负时整体取反，使「得分高 = 指标整体偏高」
    if cfg.pca_flip_auto and loading[:, 0].sum() < 0:
        eigvec[:, 0] *= -1
        loading[:, 0] *= -1
        score[:, 0] *= -1
        print("[提示] 已自动翻转第一主成分方向，使综合得分越大表示指标整体越偏高。")

    # 与 sklearn 对照：解释方差比应几乎一致（差异来自符号约定与数值精度）
    sk = SkPCA(n_components=m, random_state=cfg.random_state).fit(Z)
    diff = float(np.max(np.abs(sk.explained_variance_ratio_ - contrib)))
    print(f"\n与 sklearn PCA 的解释方差比最大差异 = {diff:.2e}（应接近 0）")

    names = [f"PC{i+1}" for i in range(m)]
    load_df = pd.DataFrame(loading, index=X.columns, columns=names).round(4)
    eig_df = pd.DataFrame(dict(主成分=names, 特征值=np.round(eigval, 4),
                               贡献率=np.round(contrib, 4), 累计贡献率=np.round(cum, 4)))
    print("\n===== 特征值与贡献率（前 10）=====")
    print(eig_df.head(10).to_string(index=False))
    return dict(z_mean=mu, z_std=sd, R=R, eigval=eigval, eigvec=eigvec,
                contrib=contrib, cum=cum, loading=loading, score=score,
                load_df=load_df, eig_df=eig_df, names=names, sk_diff=diff)


def select_components(pc, cfg):
    """按累计贡献率阈值 + Kaiser 准则给出建议主成分个数 m。"""
    kaiser = int((pc["eigval"] > 1).sum())
    m_cum = int(np.searchsorted(pc["cum"], cfg.pca_var_threshold) + 1)
    m = max(1, min(m_cum, cfg.pca_max_components, len(pc["eigval"])))
    print(f"\nKaiser 准则(λ>1) 建议 {kaiser} 个；"
          f"累计贡献率 ≥ {cfg.pca_var_threshold:.0%} 建议 {m_cum} 个；"
          f"本次采用 m = {m}")
    return m, kaiser, m_cum


def composite_score(pc, m, index):
    """综合得分 F = Σ (λ_j/Σλ) F_j，并给出排名（权重按选中主成分重新归一化）。"""
    w = pc["contrib"][:m] / pc["contrib"][:m].sum()
    F = pc["score"][:, :m] @ w
    out = pd.DataFrame(pc["score"][:, :m], index=index,
                       columns=pc["names"][:m]).round(4)
    out["综合得分"] = np.round(F, 4)
    out["排名"] = out["综合得分"].rank(ascending=False, method="min").astype(int)
    return out.sort_values("排名"), w


TEST = kmo_bartlett(X)
PC = fit_pca(X, CONFIG)
M_SEL, M_KAISER, M_CUM = select_components(PC, CONFIG)
SCORE, WEIGHTS = composite_score(PC, M_SEL, df.index)
print("\n===== 综合得分前 8 名 =====")
print(SCORE.head(8).to_string())
print("\n===== 载荷矩阵（前 6 个主成分）=====")
print(PC["load_df"].iloc[:, :min(6, M_SEL)].to_string())

===== PCA 适用性检验 =====
   KMO  Bartlett卡方  自由度    p值  KMO判断   Bartlett判断  有效秩                                                 说明
0.3812       23.9800      91 1.0000 不太适合 变量相关性不足      14 相关矩阵奇异（存在完全共线的独热列），检验值仅供参考
[提示] 适用性检验未通过：论文里应如实说明，并考虑改用熵权法 / TOPSIS，或先看相关矩阵挑出真正相关的指标组再做 PCA。

与 sklearn PCA 的解释方差比最大差异 = 9.71e-17（应接近 0）

===== 特征值与贡献率（前 10）=====
主成分  特征值  贡献率  累计贡献率
   PC1  2.2218  0.1389      0.1389
   PC2  1.8046  0.1128      0.2516
   PC3  1.5255  0.0953      0.3470
   PC4  1.4646  0.0915      0.4385
   PC5  1.2847  0.0803      0.5188
   PC6  1.2709  0.0794      0.5983
   PC7  1.0610  0.0663      0.6646
   PC8  1.0464  0.0654      0.7300
   PC9  0.9390  0.0587      0.7887
  PC10  0.8770  0.0548      0.8435

Kaiser 准则(λ>1) 建议 8 个；累计贡献率 ≥ 80% 建议 10 个；本次采用 m = 10

===== 综合得分前 8 名 =====
       PC1    PC2     PC3    PC4     PC5    PC6     PC7     PC8     PC9    PC10  综合得分  排名
177 2.3600 2.4063 -0.1049 1.8173  0.2471 1.0139 -1.0322 -0.8009  2.3417  2.3877    1.1895     1
302 2.4070 0.8914

### 6. 四张标准图与变量共同度

碎石图、载荷热图、双标图、综合得分分布；共同度低说明该变量信息没被充分保留。

In [14]:
# ============================================================
# 6. 四张标准图：碎石图 / 载荷热图 / 双标图 / 综合得分分布
# ============================================================
def plot_scree(pc, m, cfg):
    if not cfg.make_plots:
        return
    n = min(len(pc["eigval"]), 15)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    x = np.arange(1, n + 1)
    ax.bar(x, pc["eigval"][:n], color="#85B7EB", label="特征值 λ")
    ax.axhline(1, ls="--", color="#d95f5f", label="Kaiser 准则 λ=1")
    ax.set_xlabel("主成分序号"); ax.set_ylabel("特征值"); ax.set_xticks(x)
    ax2 = ax.twinx()
    ax2.plot(x, pc["cum"][:n], "o-", color="#185FA5", label="累计贡献率")
    ax2.axhline(cfg.pca_var_threshold, ls=":", color="#4a9d6a")
    ax2.set_ylabel("累计贡献率"); ax2.set_ylim(0, 1.05)
    ax2.grid(False)
    ax.set_title(f"碎石图与累计贡献率（采用 m={m}）")
    lines1, l1 = ax.get_legend_handles_labels()
    lines2, l2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, l1 + l2, fontsize=8, loc="center right")
    fig.tight_layout(); save_fig(fig, "碎石图", cfg)


def plot_loading(pc, m, cfg):
    if not cfg.make_plots:
        return
    L = pc["load_df"].iloc[:, :m]
    fig, ax = plt.subplots(figsize=(max(6, 1.0 * m), max(3.5, 0.34 * len(L))))
    im = ax.imshow(L.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(m)); ax.set_xticklabels(L.columns)
    ax.set_yticks(range(len(L))); ax.set_yticklabels(L.index)
    ax.set_title("主成分载荷矩阵（红=正相关，蓝=负相关）")
    ax.grid(False)
    fig.colorbar(im, ax=ax, label="载荷 a_ij")
    fig.tight_layout(); save_fig(fig, "载荷热图", cfg)


def plot_biplot(pc, cfg):
    """双标图：样本点（PC1/PC2）+ 变量箭头（载荷），直观解释主成分含义。"""
    if not cfg.make_plots or pc["score"].shape[1] < 2:
        return
    s = pc["score"][:, :2]
    a = pc["loading"][:, :2] * 3.0            # 箭头缩放系数，仅为可视化
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(s[:, 0], s[:, 1], s=16, alpha=0.55, color="#85B7EB")
    for i, name in enumerate(pc["load_df"].index):
        ax.annotate("", xy=(a[i, 0], a[i, 1]), xytext=(0, 0),
                    arrowprops=dict(arrowstyle="->", color="#993C1D", lw=1.1))
        ax.text(a[i, 0] * 1.05, a[i, 1] * 1.05, name, fontsize=8, color="#712B13")
    ax.axhline(0, color="#888", lw=0.6); ax.axvline(0, color="#888", lw=0.6)
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_title("双标图（样本 + 变量载荷）")
    fig.tight_layout(); save_fig(fig, "双标图", cfg)


def plot_score_dist(score, cfg):
    if not cfg.make_plots:
        return
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(score["综合得分"], bins=28, color="#4a9d6a", edgecolor="white")
    ax.axvline(score["综合得分"].mean(), ls="--", color="#d95f5f", label="均值")
    ax.set_xlabel("综合得分 F"); ax.set_ylabel("样本数")
    ax.set_title("综合得分分布"); ax.legend(fontsize=8)
    fig.tight_layout(); save_fig(fig, "综合得分", cfg)


plot_scree(PC, M_SEL, CONFIG)
plot_loading(PC, M_SEL, CONFIG)
plot_biplot(PC, CONFIG)
plot_score_dist(SCORE, CONFIG)

# 变量共同度：前 m 个主成分解释掉各变量多少信息
COMMON = pd.DataFrame(dict(
    变量=PC["load_df"].index,
    共同度=np.round((PC["loading"][:, :M_SEL] ** 2).sum(axis=1), 4)))
print("\n===== 变量共同度（越低说明该变量信息越没被保留）=====")
print(COMMON.sort_values("共同度").to_string(index=False))

[图片] PCA_示例_碎石图.png（同名覆盖）
[图片] PCA_示例_载荷热图.png（同名覆盖）
[图片] PCA_示例_双标图.png（同名覆盖）
[图片] PCA_示例_综合得分.png（同名覆盖）

===== 变量共同度（越低说明该变量信息越没被保留）=====
                 变量  共同度
         survey_score  0.4337
  temperature_missing  0.5313
         pm25_missing  0.6735
traffic_index_missing  0.8080
                 pm25  0.8725
        traffic_index  0.8744
       fuel_type=电力  0.8825
          region=南部  0.9085
          region=北部  0.9090
          region=西部  0.9213
          temperature  0.9283
 survey_score_missing  0.9316
          region=东部  0.9336
     fuel_type=天然气  0.9513
     fuel_type=生物质  0.9552
         fuel_type=煤  0.9807


### 结果怎么拿

`bundle` 装着所有中间对象，常用的几个键见下面各节；同时结果会写成
`输出数据/` 下以 `PCA` 开头、带 tag 的 `_结果明细.csv`（原表 + 预测/得分）
与 `_结果汇总.xlsx`（多 Sheet）；图片单独存 PNG。
换 tag 会新增一组文件，同 tag 是覆盖。

In [16]:
# ============================================================
# 7. 结果导出 + 新数据评分
# ============================================================
def export_results(bundle, cfg):
    pc = bundle["pca"]
    out = bundle["df"].copy()
    for j in range(min(bundle["m"], len(pc["names"]))):
        out[f"PC{j+1}"] = np.round(pc["score"][:, j], 4)
    out["综合得分"] = bundle["score"]["综合得分"].reindex(out.index).values
    out["排名"] = bundle["score"]["排名"].reindex(out.index).values
    csv = OUT_DIR / f"{PREFIX}_{cfg.tag}_结果明细.csv"
    out.to_csv(csv, index=False, encoding="utf-8-sig")

    sheets = {"数据体检": bundle["audit"],
              "适用性检验": pd.DataFrame([bundle["test"]]),
              "特征值贡献率": pc["eig_df"],
              "载荷矩阵": pc["load_df"].reset_index().rename(columns={"index": "变量"}),
              "变量共同度": bundle["communality"],
              "综合得分排名": bundle["score"].reset_index(drop=True)}
    xlsx = OUT_DIR / f"{PREFIX}_{cfg.tag}_结果汇总.xlsx"
    try:
        with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
            for n, t in sheets.items():
                t.to_excel(w, sheet_name=n[:31], index=False)
        print(f"[Excel] {xlsx.name}（{len(sheets)} 个工作表）")
    except ImportError:
        print("[提示] 缺 openpyxl，跳过 Excel；CSV 已导出。")
    print(f"[CSV]   {csv.name}")
    return csv, xlsx


def predict_new(bundle, df_new):
    """新数据评分：复用训练集的预处理器与 PCA 参数（均值/标准差/特征向量）。"""
    pc = bundle["pca"]
    Xn = bundle["preprocessor"].transform(df_new).to_numpy(float)
    Z = (Xn - pc["z_mean"]) / pc["z_std"]
    score = Z @ pc["eigvec"]
    m = bundle["m"]
    out = df_new.copy()
    for j in range(m):
        out[f"PC{j+1}"] = np.round(score[:, j], 4)
    out["综合得分"] = np.round(score[:, :m] @ bundle["weights"], 4)
    out["排名"] = out["综合得分"].rank(ascending=False, method="min").astype(int)
    print(f"新数据 {len(out)} 行完成评分")
    return out

### 一键入口 `run_pipeline` / `run_on`

`run_on("文件名", target=..., tag=...)` 是最省事的写法：自动定位文件、检查存在性、
校验参数名、创建配置、跑完整流程并返回 `bundle`。想逐项调参就用 `Config + run_pipeline`。

In [18]:
# ============================================================
# 8. 一键流程 + run_on
# ============================================================
def run_pipeline(cfg):
    print("=" * 68)
    print(f"PCA：{cfg.tag} | 数据 {cfg.data_path}")
    print("=" * 68)
    d, meta = load_table(cfg)
    aud = data_audit(d, cfg)
    p = TabularPreprocessor(cfg).fit(d)
    Xv = p.transform(d)
    test = kmo_bartlett(Xv)
    pc = fit_pca(Xv, cfg)
    m, kaiser, m_cum = select_components(pc, cfg)
    score, w = composite_score(pc, m, d.index)
    print("\n===== 综合得分前 8 名 =====")
    print(score.head(8).to_string())
    plot_scree(pc, m, cfg)
    plot_loading(pc, m, cfg)
    plot_biplot(pc, cfg)
    plot_score_dist(score, cfg)
    comm = pd.DataFrame(dict(
        变量=pc["load_df"].index,
        共同度=np.round((pc["loading"][:, :m] ** 2).sum(axis=1), 4)))

    bundle = dict(cfg=cfg, meta=meta, df=d, audit=aud, preprocessor=p, X=Xv,
                  test=test, pca=pc, m=m, m_kaiser=kaiser, m_cum=m_cum,
                  score=score, weights=w, communality=comm)
    export_results(bundle, cfg)
    print("=" * 68)
    print(f"完成：{cfg.tag} | 保留 {m} 个主成分，"
          f"累计贡献率 {pc['cum'][m-1]:.1%} | 结果见 {OUT_DIR}")
    print("=" * 68)
    return bundle


def run_on(data_path, target=None, tag="pca_demo", **overrides):
    """一行换数据：bundle = run_on("新数据.csv", tag="本次名字")"""
    path = Path(data_path)
    if not path.is_absolute():
        path = (ROOT / path) if (path.parts and path.parts[0] == DATA_DIR.name) else (DATA_DIR / path)
    if not path.exists():
        raise FileNotFoundError(f"找不到数据文件：{path}\n请把文件放进 {DATA_DIR}。")
    valid = {f.name for f in fields(Config)}
    bad = set(overrides) - valid
    if bad:
        raise ValueError(f"未知配置项 {sorted(bad)}；可用字段：{sorted(valid)}")
    params = dict(data_path=path, target=target, tag=tag)
    params.update(overrides)
    # 从当前 CONFIG 继承未显式覆盖的字段（例如逻辑回归默认丢弃独热参照组），
    # 这样「改 CONFIG 就生效」与「用 run_on 一行换数据」两条路径行为一致
    return run_pipeline(replace(CONFIG, **params))

### 主执行（默认运行这一格，就出一套完整结果）

把下面这一行的三个值换成你的文件名、标签列、tag，其余都不用动。

In [20]:
# ============================================================
# 主执行：换题目时只改这一行
# ============================================================

bundle = run_on("mcm_final_raw.csv", tag="示例")
print("bundle 可用键：", sorted(bundle.keys()))

PCA：示例 | 数据 D:\git项目\导入数据\mcm_final_raw.csv
读取成功 : mcm_final_raw.csv（utf-8-sig）
数据规模 : 381 行 → 366 行，8 列；删重复 15 行
文本转数值 : ['pm25']；日期列：['date']；哨兵转缺失：{'traffic_index': 6}
===== 数据体检表 =====
            列名           类型  缺失数  缺失率  唯一值     最小值     中位数     最大值  IQR异常点               建议
            date datetime64[us]       0  0.0000      90                                           0 日期/标识列,不建模
          region            str       0  0.0000       4                                           0           独热编码
            pm25        float64      18  0.0492     344    18.6900    50.4260    93.2070          1               保留
     temperature        float64       5  0.0137     356    11.8600    23.2560    30.6550          0               保留
   traffic_index        float64      15  0.0410     351    43.7890    98.9720   150.0030          4               保留
       fuel_type            str       8  0.0219       4                                           0           独热编码
    survey_score        flo

### 可选示例（默认不执行，把 RUN_EXAMPLES 改成 True 才会跑）

演示换数据、换参数、新数据推理；跑起来会多产出一小组文件。

In [22]:
if RUN_EXAMPLES:
    # 示例 1：换清洗后的数据，把累计贡献率阈值提到 90%，不生成图片
    b2 = run_on(OUT_DIR / "final_cleaned.csv", tag="清洗数据", make_plots=False,
                pca_var_threshold=0.90, scale="minmax")
    # 示例 2：新数据复用训练好的 PCA 参数打分
    new_score = predict_new(bundle, bundle["df"].head(8))
    print(new_score[["综合得分", "排名"]].to_string())
    # 示例 3：清理某一组历史结果
    # clean_outputs("清洗数据")
else:
    print("已跳过可选示例（RUN_EXAMPLES=False）。")

已跳过可选示例（RUN_EXAMPLES=False）。
